# Stage 6 — Signals, diversification, and walk-forward evaluation

**Self-directed academic project.** This notebook tests fixed, documented strategies and a rolling mean-variance allocation. It is not investment advice.

Momentum buys after a trailing return clears an entry threshold; mean reversion buys after price falls below its trailing mean by a z-score and exits on recovery. Each rule uses only prices known at its signal close. A one-period execution delay and a further return-period delay are applied by the engine; transaction costs and slippage are charged on all turnover and liquidation.

The first 70% of observations provide warm-up history and rolling estimates. Performance is evaluated only on the final 30%, after restarting the portfolio in cash. This is a single chronological holdout, not repeated nested validation. Markowitz weights are re-estimated from a trailing window at each rebalance and used only after the execution delay. Sample means and covariance are noisy; optimizer output should not be treated as a forecast.

In [ ]:
from portfolio_research.config import (
    load_sma_settings,
    load_strategy_settings,
    load_universe,
)
from portfolio_research.market_data import load_prices
from portfolio_research.strategies import (
    equal_weight_targets,
    mean_reversion_signals,
    momentum_signals,
    rolling_markowitz_weights,
    walk_forward_start,
)

universe = load_universe()
settings = load_strategy_settings()
trading = load_sma_settings()
data = load_prices(["SPY", "QQQ", "TLT", "GLD"], "2005-01-03", "2026-01-01")
prices = data.prices.dropna()
split = walk_forward_start(prices.index, 0.7)
prices.iloc[[0, split, -1]]

In [ ]:
spy = prices["SPY"]
momentum = momentum_signals(
    spy,
    settings.momentum_lookback_sessions,
    settings.momentum_entry_return_threshold,
    settings.momentum_exit_return_threshold,
)
mean_reversion = mean_reversion_signals(
    spy,
    settings.mean_reversion_lookback_sessions,
    settings.mean_reversion_entry_z_score,
    settings.mean_reversion_exit_z_score,
)
print("First held-out session:", prices.index[split].date())
print("Momentum target exposure:", float(momentum.iloc[split:].mean()))
print("Mean-reversion target exposure:", float(mean_reversion.iloc[split:].mean()))

In [ ]:
weights = rolling_markowitz_weights(
    prices,
    lookback_sessions=settings.markowitz_lookback_sessions,
    rebalance_every_sessions=settings.markowitz_rebalance_every_sessions,
    risk_aversion=settings.markowitz_risk_aversion,
    maximum_weight=settings.markowitz_maximum_weight,
)
display(weights.iloc[split:].dropna(how="all").head())
display(weights.iloc[split:].dropna(how="all").sum(axis=1).describe())

In [ ]:
equal_targets = equal_weight_targets(
    prices.iloc[split:],
    rebalance_every_sessions=settings.markowitz_rebalance_every_sessions,
)
display(equal_targets.dropna(how="all").head())
print("Equal weight is a benchmark, not an optimized forecast.")

## Questions to answer

1. Did Markowitz beat both SPY buy-and-hold and the rebalanced equal-weight basket after costs?
2. How much was it invested, and did that reduce some drawdowns while also missing market gains?
3. How often did Markowitz concentrate at the configured maximum weight?
4. Would changing the rule or optimizer parameters after seeing the holdout turn it into training data? Yes: if you tune after examining these results, you need a new untouched evaluation period.

Even this holdout does not remove survivorship bias, provide point-in-time constituents, or model market impact. Never search many parameter combinations and report only the winner.